# 05 — RAG Chat Logic

Where retrieval meets reasoning: take a question, pull the most relevant facts from Notebook 4's index, hand them to an LLM with strict instructions to use *only* that context, and return an answer with its sources — "The Voice" from the original pitch.

**On the model choice:** a hosted API rather than Ollama, specifically because of where this project runs. Ollama needs its own local server process and multi-gigabyte model weights — fine on a personal machine, but Colab sessions are ephemeral, so you'd re-download those weights every single time a session restarts. A hosted API has a small per-call cost, but no reinstall tax and far more reliable answers. This notebook uses Anthropic's Claude API; swapping to OpenAI is a small, contained change if you'd rather (just the `call_llm` function below).

In [11]:
!pip install ollama -q


[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Reload the index and facts

Notebook 4 already built and saved the FAISS index — no need to rebuild it, just load it back along with the embedding model (to encode new questions) and `facts.csv` (the lookup table).

In [12]:
import pandas as pd
import faiss
import ollama
from sentence_transformers import SentenceTransformer


DATA_DIR = "data"
facts_df = pd.read_csv(f"{DATA_DIR}/facts.csv")
index = faiss.read_index(f"{DATA_DIR}/facts.faiss")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Loaded {len(facts_df)} facts and an index of {index.ntotal} vectors")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3677.18it/s]


Loaded 669 facts and an index of 669 vectors


## API key

Entered through `getpass` rather than typed into a cell, so it never ends up saved in the notebook file or committed to GitHub by accident.

In [13]:
#api_key = getpass("Enter your Anthropic API key: ")
#client = anthropic.Anthropic(api_key=api_key)
# no need , because we use ollama

## Retrieval (same logic as Notebook 4)

In [14]:
MIN_SCORE = 0.35

def search(query, k=5, min_score=MIN_SCORE):
    q_vec = embed_model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_vec)
    scores, ids = index.search(q_vec, k)
    return [
        {"text": facts_df.iloc[i]["text"], "category": facts_df.iloc[i]["category"],
         "n": int(facts_df.iloc[i]["n"]), "score": float(s)}
        for i, s in zip(ids[0], scores[0]) if s >= min_score
    ]

## The prompt

The system instruction is the whole reason this is "RAG" rather than just an LLM guessing: it's told to answer *only* from the facts it's given, and to say so honestly when the facts don't fully cover the question, rather than filling gaps with plausible-sounding invention.

In [15]:
SYSTEM_PROMPT = (
    "You are a career advisor for software developers. Answer using ONLY the survey "
    "facts provided in the context below — never use outside knowledge or assumptions. "
    "Each fact states its sample size (n); weigh facts from larger samples more heavily, "
    "and mention when a number rests on a small sample. If the provided facts don't fully "
    "answer the question, say so plainly rather than guessing at the rest."
)

def build_user_message(question, retrieved_facts):
    context = "\n".join(f"- {f['text']}" for f in retrieved_facts)
    return f"Context (2025 Stack Overflow Developer Survey):\n{context}\n\nQuestion: {question}"

## Tying it together

In [16]:
def ask(question, k=5):
    retrieved = search(question, k=k)
    if not retrieved:
        return {
            "answer": "I don't have survey data that speaks to that directly — I can only answer from the 2025 Stack Overflow Developer Survey facts I've indexed.",
            "sources": [],
        }
    user_message = build_user_message(question, retrieved)
    response = ollama.chat(
    model="llama3.2",
    messages=[
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_message
        }
    ]
)
    return {"answer": response["message"]["content"],"sources": retrieved}


def show(question):
    result = ask(question)
    print(f"Q: {question}\n")
    print(f"A: {result['answer']}\n")
    if result["sources"]:
        print("Sources:")
        for s in result["sources"]:
            print(f"  [{s['category']}, n={s['n']}] {s['text'][:80]}")
    print("\n" + "="*80 + "\n")

## Test it — starting with your own example from the pitch

In [17]:
show("I'm a junior dev in India learning Python. Should I learn Rust or Go next for the best salary bump?")

Q: I'm a junior dev in India learning Python. Should I learn Rust or Go next for the best salary bump?

A: Based on the survey facts provided, it's difficult to make an informed decision without knowing more about your personal goals and preferences. However, considering the number of respondents (2020, 809, 4695, 514, and 2795), we can look at the proportion of developers using Python in each country.

Since you're from India, the survey fact that 57% of Indian developers used Python in the past year is relevant. However, this figure only represents respondents from 2020, which might not reflect your current situation or be representative of the entire population of junior devs in India (no sample size is provided for this group).

As for learning Rust or Go next for a salary bump, we don't have any survey facts directly comparing these languages. However, considering the fact that among DevOps engineers or professionals, 77% used Python as their language/technology in the past year, 

In [18]:
show("Is a CS degree worth it compared to being self-taught?")
show("What is the best pizza topping?")

Q: Is a CS degree worth it compared to being self-taught?

A: This question cannot be directly answered using the provided survey facts. The survey only provides information about average job satisfaction by education level, but does not explicitly compare the two options (CS degree vs self-taught).

Sources:
  [satisfaction_by_education, n=3124] Among developers whose highest education is 'Some college/university study witho
  [satisfaction_by_education, n=821] Among developers whose highest education is 'Associate degree (A.A., A.S., etc.)


Q: What is the best pizza topping?

A: I don't have survey data that speaks to that directly — I can only answer from the 2025 Stack Overflow Developer Survey facts I've indexed.





## What we built

- Retrieval and generation fully wired together: a question becomes a search, the results become grounded context, and the LLM answers strictly from that context
- The out-of-scope test should show the honest "no data" response rather than a hallucinated guess — worth double-checking when you run this for real
- Every answer comes with its sources shown, matching the "Sources Cited" UX from the original pitch

**One thing I couldn't verify from here:** the actual `client.messages.create` call needs a real API key and a live network call, neither of which I can do from this sandbox. The code follows the current, documented Anthropic SDK pattern, but you'll be the one to see the real answers come back.

**Next:** Notebook 5 lives in a notebook — the last step is porting this same `ask()` logic into `app.py`, a standalone Streamlit script, for the actual chat interface.